## Cell 1


In [ ]:
%pip install -q faiss-cpu sentence-transformers

## Cell 2


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cpu"
)

documents = [
    "Machine learning allows computers to learn from data.",
    "Deep learning uses neural networks.",
    "Python is a popular programming language.",
    "Vector databases store numerical embeddings.",
    "FAISS performs efficient similarity searches.",
    "HNSW is a graph-based nearest neighbour algorithm."
]

embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

## Cell 3


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Vectors stored:", index.ntotal)

## Cell 4


In [ ]:
query = "How do computers learn from information?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

distances, indices = index.search(query_embedding, k=3)

for position, score in zip(indices[0], distances[0]):
    print("Document:", documents[position])
    print("Similarity:", round(float(score), 4))
    print()

## Cell 5


In [ ]:
dimension = embeddings.shape[1]

index_hnsw = faiss.IndexHNSWFlat(
    dimension,
    32,
    faiss.METRIC_INNER_PRODUCT
)

index_hnsw.hnsw.efConstruction = 100
index_hnsw.hnsw.efSearch = 50

index_hnsw.add(embeddings)

print("Vectors stored:", index_hnsw.ntotal)

## Cell 6


In [ ]:
distances, indices = index_hnsw.search(
    query_embedding,
    k=3
)

for position, score in zip(indices[0], distances[0]):
    print("Document:", documents[position])
    print("Similarity:", round(float(score), 4))
    print()

## Cell 7


In [ ]:
%pip install -q chromadb sentence-transformers

## Cell 8


In [ ]:
import chromadb

print("ChromaDB version:", chromadb.__version__)
print("ChromaDB imported successfully!")

## Cell 9


In [ ]:
import chromadb

client = chromadb.PersistentClient(
    path="/content/hnsw_database"
)

collection = client.create_collection(
    name="hnsw_books",
    configuration={
        "hnsw": {
            "space": "cosine",
            "ef_construction": 100,
            "max_neighbors": 16,
            "ef_search": 50
        }
    }
)

print("HNSW collection created!")

## Cell 10


In [ ]:
collection.add(
    ids=[f"book{i}" for i in range(len(documents))],
    documents=documents
)

results = collection.query(
    query_texts=["How do computers learn from information?"],
    n_results=3
)

for document in results["documents"][0]:
    print(document)

## Cell 11


In [ ]:
documents = [
    "The Hobbit is a fantasy novel about Bilbo Baggins.",
    "1984 is a dystopian novel by George Orwell.",
    "Pride and Prejudice explores love and social class.",
    "Dune is a science fiction novel about a desert planet.",
    "Animal Farm is a political allegory about totalitarianism.",
    "Harry Potter follows a young wizard attending Hogwarts."
]

collection.add(
    ids=["book1", "book2", "book3", "book4", "book5", "book6"],
    documents=documents
)

print("Books added:", collection.count())

## Cell 12


In [ ]:
for document, distance in zip(
    results["documents"][0],
    results["distances"][0]
):
    print("Book:", document)
    print("Distance:", round(distance, 4))
    print()

## Cell 13


In [ ]:
results = collection.query(
    query_texts=["A story about an authoritarian government"],
    n_results=3
)

for document in results["documents"][0]:
    print(document)